# Week 3: First Quantum Kernel Results
**Project**: Quantum Kernels on CLIP/BERT Embeddings

**What this notebook does:**
1. Loads the Week 2 saved embeddings (no re-running BERT)
2. Implements an IQP quantum feature map with 8 qubits
3. Computes the quantum kernel matrix and trains a quantum kernel SVM
4. Compares directly against the Week 2 classical baselines
5. Produces the first comparison figure for the paper

**Baselines to beat (from Week 2):**
- Linear SVM: 67.0% accuracy, F1=0.708
- RBF SVM: 64.0% accuracy, F1=0.673
- Polynomial SVM: 65.0% accuracy, F1=0.647

**Runtime**: ~5 min for N=200 (recommended), ~20 min for N=500 (paper runs)

In [ ]:
!pip install pennylane pennylane-lightning --quiet

import pennylane as qml
import numpy as np
import time, pickle
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

print(f"PennyLane version: {qml.__version__}")
print("All imports OK")

---
## Step 1: Load Week 2 data

We load the pre-computed embeddings directly — no need to re-run BERT.
Upload the three files from your `qml_project/` folder before running this cell:
- `embeddings_week2.npz`
- `baseline_results.pkl`

In [ ]:
# Upload files to Colab first:
# Runtime > Upload files  OR  use the Files panel on the left
# Then run this cell

data = np.load('embeddings_week2.npz')

X_tr_full = data['X_tr_subset'].astype(np.float64)   # (500, 8)
X_te_full = data['X_te_subset'].astype(np.float64)   # (100, 8)
y_tr_full = data['y_tr_subset']
y_te_full = data['y_te_subset']

with open('baseline_results.pkl', 'rb') as f:
    baselines = pickle.load(f)

print(f"Training subset : {X_tr_full.shape}  labels: {X_tr_full.shape[0]}")
print(f"Test subset     : {X_te_full.shape}  labels: {X_te_full.shape[0]}")
print(f"Value range     : [{X_tr_full.min():.4f}, {X_tr_full.max():.4f}]  (must be in [0, pi])")
print(f"Class balance   : {np.mean(y_tr_full==1):.1%} positive in train, {np.mean(y_te_full==1):.1%} in test")
print()
print("Classical baselines loaded:")
for name, vals in baselines.items():
    print(f"  {name:<14} acc={vals['accuracy']:.4f}  f1={vals['f1']:.4f}")

---
## Step 2: Quantum feature map (IQP circuit)

We use an **IQP (Instantaneous Quantum Polynomial)** feature map — the same design  
used in Havlíček et al. 2019, the paper that started quantum kernel research.

The circuit structure for input vector x = [x₀, x₁, ..., x₇]:
1. **Hadamard** on all qubits → puts them in superposition
2. **RZ(xᵢ)** on each qubit → encodes each feature as a rotation angle
3. **CNOT + RZ(xᵢ·xⱼ) + CNOT** between adjacent pairs → encodes cross-feature interactions

The kernel value K(x₁,x₂) = probability of measuring |00000000⟩ after applying  
U(x₁) then U†(x₂) — the "overlap" between the two encoded quantum states.

In [ ]:
N_QUBITS = 8
dev = qml.device('lightning.qubit', wires=N_QUBITS)  # faster simulator

def iqp_feature_map(x, wires):
    """IQP feature map: Hadamard + angle encoding + ZZ entanglement."""
    n = len(wires)
    # Layer 1: superposition
    for w in wires:
        qml.Hadamard(wires=w)
    # Layer 2: single-qubit feature encoding
    for i, w in enumerate(wires):
        qml.RZ(x[i], wires=w)
    # Layer 3: two-qubit ZZ interactions (encodes x_i * x_j cross terms)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i] * x[i+1], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

@qml.qnode(dev)
def kernel_circuit(x1, x2):
    """K(x1, x2) = |<0|U†(x2)·U(x1)|0>|²"""
    iqp_feature_map(x1, wires=range(N_QUBITS))
    qml.adjoint(iqp_feature_map)(x2, wires=range(N_QUBITS))
    return qml.probs(wires=range(N_QUBITS))

def quantum_kernel(x1, x2):
    """Returns scalar kernel value — probability of |000...0> state."""
    return float(kernel_circuit(x1, x2)[0])

# Sanity checks
x_a = X_tr_full[0]
x_b = X_tr_full[1]
k_same = quantum_kernel(x_a, x_a)
k_diff = quantum_kernel(x_a, x_b)
k_sym  = quantum_kernel(x_b, x_a)

print(f"Sanity checks:")
print(f"  K(x, x)   = {k_same:.6f}  (expected: 1.0)")
print(f"  K(x1, x2) = {k_diff:.6f}  (expected: < 1.0)")
print(f"  K(x2, x1) = {k_sym:.6f}  (symmetric: {np.isclose(k_diff, k_sym, atol=1e-5)})")
print()

# Benchmark single eval time
t0 = time.time()
for _ in range(5):
    quantum_kernel(x_a, x_b)
ms = (time.time() - t0) / 5 * 1000
print(f"Single kernel eval: {ms:.1f} ms")
print()
for n in [50, 100, 200, 500]:
    est = n * n * ms / 1000 / 60
    print(f"  N={n:4d} kernel matrix: ~{est:.1f} min")

---
## Step 3: Quick test on N=50

Before committing to the full run, we validate on 50 samples.  
This takes ~1–2 minutes and confirms the pipeline produces sensible results.

In [ ]:
N_QUICK = 50

X_tr_q = X_tr_full[:N_QUICK]
y_tr_q = y_tr_full[:N_QUICK]
X_te_q = X_te_full[:20]
y_te_q = y_te_full[:20]

print(f"Quick test: {N_QUICK} train, 20 test")
print("Computing kernel matrices...")
t0 = time.time()
K_tr_q  = qml.kernels.kernel_matrix(X_tr_q, X_tr_q, quantum_kernel)
K_te_q  = qml.kernels.kernel_matrix(X_te_q, X_tr_q, quantum_kernel)
elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s")

# Validate kernel matrix properties
eigvals = np.linalg.eigvalsh(K_tr_q)
print(f"\nKernel matrix validation:")
print(f"  Symmetric          : {np.allclose(K_tr_q, K_tr_q.T, atol=1e-6)}")
print(f"  Positive semi-def  : {eigvals.min() >= -1e-6} (min eigenvalue: {eigvals.min():.6f})")
print(f"  Diagonal values    : mean={np.diag(K_tr_q).mean():.4f} (should be ~1.0)")

# Quick SVM
svm_q = SVC(kernel='precomputed', C=1.0)
svm_q.fit(K_tr_q, y_tr_q)
acc_q = accuracy_score(y_te_q, svm_q.predict(K_te_q))
print(f"\nQuick test accuracy (N=50, 20 test): {acc_q:.4f}")
print("Pipeline confirmed working. Proceeding to full run...")

# Visualize the 50x50 kernel matrix
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(K_tr_q, cmap='Blues', vmin=0, vmax=1)
axes[0].set_title(f'Quantum kernel matrix (N=50)')
axes[0].set_xlabel('Sample j'); axes[0].set_ylabel('Sample i')
plt.colorbar(im0, ax=axes[0])

# Compare with RBF kernel matrix on same data
from sklearn.metrics.pairwise import rbf_kernel
K_rbf_q = rbf_kernel(X_tr_q, gamma=1.0/N_QUBITS)
im1 = axes[1].imshow(K_rbf_q, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title(f'RBF kernel matrix (N=50, for comparison)')
axes[1].set_xlabel('Sample j')
plt.colorbar(im1, ax=axes[1])
plt.suptitle('Kernel matrix comparison — quantum vs RBF (same 50 samples)')
plt.tight_layout()
plt.show()

---
## Step 4: Full run on N=200

This is the main experimental result for the paper.  
**Expected time: ~3–5 minutes on Colab.**

We use 200 training samples — enough to show meaningful patterns  
without the 20-minute wait of the full N=500 run.

In [ ]:
N_MAIN = 200

X_tr_m = X_tr_full[:N_MAIN]
y_tr_m = y_tr_full[:N_MAIN]
X_te_m = X_te_full          # all 100 test samples
y_te_m = y_te_full

print(f"Main run: N={N_MAIN} train, {len(X_te_m)} test")
print("Computing quantum kernel matrix (this takes a few minutes)...")
print("Progress bar below:")

t0 = time.time()
K_train = qml.kernels.kernel_matrix(X_tr_m, X_tr_m, quantum_kernel)
K_test  = qml.kernels.kernel_matrix(X_te_m, X_tr_m, quantum_kernel)
elapsed = time.time() - t0
print(f"\nDone in {elapsed/60:.1f} min  ({elapsed:.0f}s)")

# Train quantum kernel SVM
qsvm = SVC(kernel='precomputed', C=1.0)
qsvm.fit(K_train, y_tr_m)
y_pred_q = qsvm.predict(K_test)

acc_q  = accuracy_score(y_te_m, y_pred_q)
f1_q   = f1_score(y_te_m, y_pred_q)

print(f"\n{'='*50}")
print(f"QUANTUM KERNEL SVM (N=200, 8 qubits, IQP):")
print(f"  Accuracy : {acc_q:.4f}")
print(f"  F1 score : {f1_q:.4f}")
print(f"{'='*50}")
print()
print("vs Classical baselines (N=500):")
for name, vals in baselines.items():
    diff = acc_q - vals['accuracy']
    sign = '+' if diff >= 0 else ''
    print(f"  vs {name:<12}: {sign}{diff:+.4f}  ({sign}{diff*100:.1f}%)")

---
## Step 5: Retrain classical baselines at N=200 (fair comparison)

The classical baselines in `baseline_results.pkl` were trained on N=500.  
For a fair apples-to-apples comparison, we retrain them on N=200 too.

In [ ]:
from sklearn.svm import SVC as ClassicSVC

classical_200 = {
    'Linear':     ClassicSVC(kernel='linear',  C=1.0, random_state=42),
    'RBF':        ClassicSVC(kernel='rbf',     C=1.0, gamma='scale', random_state=42),
    'Polynomial': ClassicSVC(kernel='poly',    C=1.0, degree=3, coef0=1, random_state=42),
}

results_200 = {}
print(f"Classical SVMs retrained on N=200:")
print(f"  {'Kernel':<14} {'Acc':>8}  {'F1':>8}")
print(f"  {'-'*34}")

for name, clf in classical_200.items():
    clf.fit(X_tr_m, y_tr_m)
    y_pred = clf.predict(X_te_m)
    acc = accuracy_score(y_te_m, y_pred)
    f1  = f1_score(y_te_m, y_pred)
    results_200[name] = {'accuracy': acc, 'f1': f1}
    print(f"  {name:<14} {acc:>8.4f}  {f1:>8.4f}")

results_200['Quantum (IQP)'] = {'accuracy': acc_q, 'f1': f1_q}
print(f"  {'Quantum (IQP)':<14} {acc_q:>8.4f}  {f1_q:>8.4f}")

---
## Step 6: Comparison figure (paper Figure 1)

This is the main figure for the paper — all methods on the same axis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

names  = list(results_200.keys())
accs   = [results_200[n]['accuracy'] for n in names]
f1s    = [results_200[n]['f1'] for n in names]
colors = ['#5B8DB8', '#5B8DB8', '#5B8DB8', '#D95F02']  # quantum highlighted

# Accuracy
bars0 = axes[0].bar(names, accs, color=colors, edgecolor='white', linewidth=1.5, width=0.6)
axes[0].set_ylim(0.5, 0.85)
axes[0].set_title('Accuracy — N=200 training samples', fontsize=13, pad=10)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].axhline(y=max(accs[:3]), color='#5B8DB8', linestyle=':', alpha=0.4, label='Best classical')
for bar, val in zip(bars0, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.005,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='500')
axes[0].set_xticklabels(names, rotation=15, ha='right')

# F1
bars1 = axes[1].bar(names, f1s, color=colors, edgecolor='white', linewidth=1.5, width=0.6)
axes[1].set_ylim(0.5, 0.85)
axes[1].set_title('F1 score — N=200 training samples', fontsize=13, pad=10)
axes[1].set_ylabel('F1 score', fontsize=11)
axes[1].axhline(y=max(f1s[:3]), color='#5B8DB8', linestyle=':', alpha=0.4, label='Best classical')
for bar, val in zip(bars1, f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.005,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='500')
axes[1].set_xticklabels(names, rotation=15, ha='right')

blue_patch  = mpatches.Patch(color='#5B8DB8', label='Classical kernels')
orange_patch = mpatches.Patch(color='#D95F02', label='Quantum kernel (IQP, 8 qubits)')
fig.legend(handles=[blue_patch, orange_patch], loc='upper center',
           ncol=2, fontsize=11, bbox_to_anchor=(0.5, 1.02))

plt.suptitle('Quantum vs classical SVM kernels on 8-dim BERT embeddings (SST-2)',
             fontsize=12, y=1.06)
plt.tight_layout()
plt.savefig('fig1_comparison.pdf', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: fig1_comparison.pdf")

---
## Step 7: Learning curve — accuracy vs training size

This is a key analysis for the paper. It shows whether quantum kernels are  
more data-efficient than classical ones — a plausible advantage in the NISQ regime.

In [ ]:
train_sizes = [25, 50, 75, 100, 150, 200]
lc_results = {'Quantum (IQP)': [], 'Linear': [], 'RBF': []}

print("Computing learning curves (this will take a few minutes)...")
print(f"  {'N':>5}  {'Quantum':>10}  {'Linear':>10}  {'RBF':>10}  {'Time':>8}")
print(f"  {'-'*50}")

for n in train_sizes:
    X_n = X_tr_full[:n]
    y_n = y_tr_full[:n]
    t0 = time.time()

    # Quantum
    K_n      = qml.kernels.kernel_matrix(X_n, X_n, quantum_kernel)
    K_n_test = qml.kernels.kernel_matrix(X_te_m, X_n, quantum_kernel)
    svm_n    = SVC(kernel='precomputed', C=1.0).fit(K_n, y_n)
    acc_qn   = accuracy_score(y_te_m, svm_n.predict(K_n_test))
    lc_results['Quantum (IQP)'].append(acc_qn)

    # Classical
    for cname, ck in [('Linear','linear'), ('RBF','rbf')]:
        clf_n = ClassicSVC(kernel=ck, C=1.0, gamma='scale' if ck=='rbf' else 1.0,
                           random_state=42).fit(X_n, y_n)
        lc_results[cname].append(accuracy_score(y_te_m, clf_n.predict(X_te_m)))

    elapsed_n = time.time() - t0
    print(f"  {n:>5}  {lc_results['Quantum (IQP)'][-1]:>10.4f}  "
          f"{lc_results['Linear'][-1]:>10.4f}  {lc_results['RBF'][-1]:>10.4f}  "
          f"{elapsed_n:>6.0f}s")

# Plot
plt.figure(figsize=(8, 4))
colors_lc = {'Quantum (IQP)': '#D95F02', 'Linear': '#5B8DB8', 'RBF': '#2D9C4A'}
for method, accs_lc in lc_results.items():
    ls = '-' if method == 'Quantum (IQP)' else '--'
    lw = 2.5 if method == 'Quantum (IQP)' else 1.8
    plt.plot(train_sizes, accs_lc, label=method, color=colors_lc[method],
             linestyle=ls, linewidth=lw, marker='o', markersize=5)

plt.xlabel('Training set size (N)', fontsize=11)
plt.ylabel('Test accuracy', fontsize=11)
plt.title('Learning curves — quantum vs classical SVM\n(8-dim BERT embeddings, SST-2)', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig2_learning_curves.pdf', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: fig2_learning_curves.pdf")

---
## Step 8: Save Week 3 results

Save everything for Week 4 (ablation study).

In [ ]:
import json

week3_results = {
    'n_qubits': N_QUBITS,
    'n_train': N_MAIN,
    'n_test': len(X_te_m),
    'feature_map': 'IQP (Hadamard + RZ + ZZ)',
    'quantum_accuracy': float(acc_q),
    'quantum_f1': float(f1_q),
    'all_results_n200': {k: {'accuracy': float(v['accuracy']), 'f1': float(v['f1'])}
                         for k, v in results_200.items()},
    'learning_curve_sizes': train_sizes,
    'learning_curve_results': {k: [float(v) for v in vals]
                                for k, vals in lc_results.items()},
}

with open('results_week3.json', 'w') as f:
    json.dump(week3_results, f, indent=2)

# Save kernel matrices for Week 4 analysis
np.savez('kernel_matrices_week3.npz',
         K_train_quantum=K_train,
         K_test_quantum=K_test,
         X_tr=X_tr_m,
         y_tr=y_tr_m)

print("Saved:")
print("  results_week3.json        — accuracy/F1 table + learning curves")
print("  kernel_matrices_week3.npz — quantum kernel matrices for Week 4")
print()
print("=== WEEK 3 SUMMARY ===")
print(f"Quantum kernel (IQP, 8 qubits, N=200):")
print(f"  Accuracy : {acc_q:.4f}")
print(f"  F1 score : {f1_q:.4f}")
print()
print("Next: Week 4 — sweep qubit counts (4/8/12) and feature maps")

---
## Week 3 complete

### What you have now:
- First quantum kernel result on real BERT embeddings
- Direct comparison against 3 classical baselines at N=200
- Learning curve showing data efficiency across training sizes
- Two paper-ready figures saved as PDF

### Key question to answer in Week 4:
Does the quantum kernel improve with more qubits?  
Does a different feature map help?  
Is there a training-size regime where quantum clearly wins or loses?

### Files to carry into Week 4:
- `results_week3.json` — your Week 3 numbers
- `kernel_matrices_week3.npz` — quantum kernel matrices for alignment analysis
- `embeddings_week2.npz` — still needed for ablations